# 🎓 Smart Learning — AI Classroom Vision Engine (Google Colab T4 GPU)
**Tim SYNTAX SLAYER (Institut Teknologi Garut)**

Notebook ini menjalankan Computer Vision (YOLOv8-Pose 17-Keypoints & Behavior Telemetry) di atas **Nvidia T4 GPU (16 GB VRAM) Gratis** milik Google Colab, lalu memancarkan URL publik HTTPS via Cloudflare Tunnel.

In [ ]:
# 1. Verifikasi Akselerasi GPU
!nvidia-smi

In [ ]:
# 2. Install Dependensi AI & Web Server
!pip install -q ultralytics opencv-python-headless fastapi uvicorn requests pydantic
!curl -s -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared.deb > /dev/null 2>&1
print('✅ Dependensi & Cloudflared berhasil diinstall!')

In [ ]:
# 3. Script Engine Vision & API Server (app.py)
code = '''import asyncio
import json
import logging
import math
import os
import queue
import random
import threading
import time
from datetime import datetime, timezone
from typing import Optional

import cv2
import numpy as np
import requests
from fastapi import FastAPI, Request, Response
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse, JSONResponse, StreamingResponse
from pydantic import BaseModel

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("HF-Classroom-AI")

# Ultralytics Import with safe fallback
HAS_ULTRALYTICS = False
try:
    from ultralytics import YOLO
    HAS_ULTRALYTICS = True
    logger.info("Ultralytics YOLO imported successfully.")
except ImportError:
    logger.warning("Ultralytics not available, falling back to heuristic/synthetic engine.")

app = FastAPI(title="Smart Learning AI Edge Engine")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Colors (BGR)
COLOR_NEON_YELLOW = (40, 235, 255)
COLOR_ATTENTIVE = (80, 210, 100)
COLOR_WRITING = (255, 180, 60)
COLOR_PHONE = (40, 40, 245)
COLOR_DISTRACTED = (30, 150, 255)
COLOR_SLEEP = (50, 50, 240)
COLOR_HAND = (40, 170, 255)

# Global State
class EngineState:
    def __init__(self):
        self.source = os.getenv("CAMERA_SOURCE", "simulate")
        self.api_url = os.getenv("API_URL", "http://localhost:3000/api/telemetry/ingest")
        self.device_key = os.getenv("DEVICE_KEY", "dev-edge-secret-key-2026")
        self.session_id = os.getenv("SESSION_ID", "00000000-0000-0000-0000-000000000001")
        self.latest_frame_jpeg: Optional[bytes] = None
        self.latest_telemetry = {
            "status": "Initializing",
            "source": self.source,
            "fps": 0.0,
            "student_count": 0,
            "attentive_count": 0,
            "hand_raised_count": 0,
            "sleeping_count": 0,
            "average_focus_score": 85.0,
            "timestamp": datetime.now(timezone.utc).isoformat()
        }
        self.lock = threading.Lock()
        self.stop_event = threading.Event()
        self.cap = None

state = EngineState()

def create_synthetic_frame(tick: int):
    """Menghasilkan frame simulasi kelas berspesifikasi tinggi jika kamera offline."""
    frame = np.zeros((720, 1280, 3), dtype=np.uint8)
    frame[:] = (24, 27, 32) # Dark background

    # Title header
    cv2.putText(frame, "SMART LEARNING - CLOUD VISION RUNTIME [SIMULATED]", (30, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (200, 200, 200), 2)
    cv2.putText(frame, f"Source: {state.source} | Mode: AI Keypoint Analysis", (30, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (140, 140, 140), 1)

    # Grid meja simulasi 6 siswa
    students = [
        {"name": "Zaldy M. F.", "seat": "M-01", "x": 160, "y": 200, "state": "ATTENTIVE"},
        {"name": "Siti Nurhaliza", "seat": "M-02", "x": 520, "y": 200, "state": "WRITING"},
        {"name": "Ahmad Fauzi", "seat": "M-03", "x": 880, "y": 200, "state": "HAND_RAISED"},
        {"name": "Dewi Sartika", "seat": "M-04", "x": 160, "y": 440, "state": "ATTENTIVE"},
        {"name": "Budi Santoso", "seat": "M-05", "x": 520, "y": 440, "state": "SLEEP" if (tick // 30) % 2 == 0 else "ATTENTIVE"},
        {"name": "Rina Marlina", "seat": "M-06", "x": 880, "y": 440, "state": "WRITING"},
    ]

    attentive_cnt = 0
    hand_cnt = 0
    sleep_cnt = 0

    for st in students:
        sx, sy = st["x"], st["y"]
        w, h = 260, 190
        status = st["state"]

        if status == "ATTENTIVE":
            color = COLOR_ATTENTIVE
            attentive_cnt += 1
        elif status == "WRITING":
            color = COLOR_WRITING
            attentive_cnt += 1
        elif status == "HAND_RAISED":
            color = COLOR_HAND
            hand_cnt += 1
        else:
            color = COLOR_SLEEP
            sleep_cnt += 1

        # Card / Desk Box
        cv2.rectangle(frame, (sx, sy), (sx + w, sy + h), (40, 45, 52), -1)
        cv2.rectangle(frame, (sx, sy), (sx + w, sy + h), color, 2)

        # Skeleton stickman avatar
        head_x, head_y = sx + w // 2, sy + 60
        if status == "SLEEP":
            head_y += 35
        cv2.circle(frame, (head_x, head_y), 22, color, 2)
        cv2.line(frame, (head_x, head_y + 22), (head_x, sy + 140), COLOR_NEON_YELLOW, 3)

        # Hands
        if status == "HAND_RAISED":
            cv2.line(frame, (head_x, head_y + 35), (head_x + 45, sy + 20), COLOR_NEON_YELLOW, 3)
            cv2.circle(frame, (head_x + 45, sy + 20), 6, COLOR_HAND, -1)
        else:
            cv2.line(frame, (head_x, head_y + 35), (head_x + 35, sy + 120), COLOR_NEON_YELLOW, 2)

        cv2.line(frame, (head_x, head_y + 35), (head_x - 35, sy + 120), COLOR_NEON_YELLOW, 2)

        # Nameplate
        cv2.rectangle(frame, (sx, sy), (sx + w, sy + 28), color, -1)
        cv2.putText(frame, f"{st['seat']} - {st['name']}", (sx + 8, sy + 19),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (10, 10, 10), 2)
        cv2.putText(frame, f"STATUS: {status}", (sx + 10, sy + h - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, color, 1)

    # Telemetry HUD
    cv2.rectangle(frame, (30, 660), (1250, 705), (15, 17, 21), -1)
    hud_text = f"ATTENTIVE: {attentive_cnt} | HAND RAISED: {hand_cnt} | SLEEP: {sleep_cnt} | TOTAL: {len(students)}"
    cv2.putText(frame, hud_text, (50, 692), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 180), 2)

    with state.lock:
        state.latest_telemetry = {
            "status": "Simulated Active",
            "source": state.source,
            "fps": 30.0,
            "student_count": len(students),
            "attentive_count": attentive_cnt,
            "hand_raised_count": hand_cnt,
            "sleeping_count": sleep_cnt,
            "average_focus_score": round(82.5 + random.uniform(-2, 3), 1),
            "timestamp": datetime.now(timezone.utc).isoformat()
        }

    return frame

def vision_worker():
    """Background thread running video capture and YOLOv8 inference."""
    logger.info("Starting Vision Worker Thread...")
    yolo_model = None
    if HAS_ULTRALYTICS:
        try:
            yolo_model = YOLO("yolov8n-pose.pt")
            logger.info("YOLOv8n-pose loaded successfully in worker.")
        except Exception as e:
            logger.error(f"Error loading YOLO: {e}")

    tick = 0
    last_source = None
    cap = None

    while not state.stop_event.is_set():
        current_source = state.source

        if current_source != last_source:
            if cap is not None:
                cap.release()
                cap = None
            if current_source not in ("simulate", "0", "") and current_source.startswith("http"):
                logger.info(f"Connecting to video stream source: {current_source}")
                cap = cv2.VideoCapture(current_source)
            last_source = current_source

        frame = None
        if cap is not None and cap.isOpened():
            ret, captured = cap.read()
            if ret and captured is not None:
                frame = captured
            else:
                logger.warning("Stream frame drop, switching to fallback...")
                time.sleep(0.1)

        if frame is None:
            # Fallback to simulation
            tick += 1
            frame = create_synthetic_frame(tick)
            time.sleep(0.04) # ~25 FPS
        else:
            # Run YOLOv8 Pose on real camera frame
            if yolo_model is not None:
                try:
                    results = yolo_model(frame, verbose=False)
                    frame = results[0].plot()
                except Exception as e:
                    logger.error(f"Inference error: {e}")

        # Encode to JPEG
        ret, jpeg = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 75])
        if ret:
            with state.lock:
                state.latest_frame_jpeg = jpeg.tobytes()

@app.on_event("startup")
def startup_event():
    t = threading.Thread(target=vision_worker, daemon=True)
    t.start()

class SourcePayload(BaseModel):
    source: str

@app.post("/api/set_source")
def set_source(payload: SourcePayload):
    with state.lock:
        state.source = payload.source
    logger.info(f"Source updated to: {payload.source}")
    return {"success": True, "source": state.source}

@app.get("/api/telemetry")
def get_telemetry():
    with state.lock:
        return state.latest_telemetry

def gen_mjpeg():
    while True:
        with state.lock:
            frame_data = state.latest_frame_jpeg
        if frame_data is not None:
            yield (b"--frame\\r\\n"
                   b"Content-Type: image/jpeg\\r\\n\\r\\n" + frame_data + b"\\r\\n")
        time.sleep(0.04)

@app.get("/video_feed")
def video_feed():
    return StreamingResponse(gen_mjpeg(), media_type="multipart/x-mixed-replace; boundary=frame")

@app.get("/", response_class=HTMLResponse)
def index():
    return """
<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Smart Learning AI Edge - Hugging Face</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <style>
        body { background-color: #0b0f17; color: #f1f5f9; font-family: ui-sans-serif, system-ui; }
    </style>
</head>
<body class="p-6 max-w-6xl mx-auto">
    <div class="flex items-center justify-between pb-6 border-b border-slate-800">
        <div>
            <span class="text-xs font-semibold px-2.5 py-0.5 rounded bg-emerald-950 text-emerald-400 border border-emerald-800">HF Spaces Cloud Engine</span>
            <h1 class="text-2xl font-bold mt-1 text-slate-100">🎓 Smart Learning — AI Classroom Vision</h1>
            <p class="text-sm text-slate-400">Tim SYNTAX SLAYER • Edge YOLOv8 Pose + Behavioral Telemetry</p>
        </div>
        <div class="text-right">
            <span id="badge-status" class="inline-flex items-center gap-1.5 px-3 py-1 rounded-full text-xs font-medium bg-emerald-900/60 text-emerald-300">
                <span class="w-2 h-2 rounded-full bg-emerald-400 animate-pulse"></span> Streaming Active
            </span>
        </div>
    </div>

    <!-- Main Grid -->
    <div class="grid grid-cols-1 lg:grid-cols-3 gap-6 mt-6">
        <!-- Live Video View -->
        <div class="lg:col-span-2 bg-slate-900 border border-slate-800 rounded-xl overflow-hidden shadow-2xl">
            <div class="p-3 bg-slate-950 border-b border-slate-800 flex justify-between items-center text-xs text-slate-400">
                <span>🔴 LIVE STREAM (MJPEG :7860/video_feed)</span>
                <span id="stat-fps">30 FPS</span>
            </div>
            <div class="relative bg-black aspect-video flex items-center justify-center">
                <img src="/video_feed" class="w-full h-full object-contain" alt="Live Feed" />
            </div>
        </div>

        <!-- Controls & Metrics -->
        <div class="space-y-6">
            <!-- Camera Source Switcher -->
            <div class="bg-slate-900 border border-slate-800 rounded-xl p-5">
                <h3 class="text-sm font-semibold text-slate-200 mb-3">📡 Ubah Sumber Video (CCTV / HP)</h3>
                <p class="text-xs text-slate-400 mb-3">Masukkan URL IP Webcam HP (via Ngrok/Cloudflare) atau RTSP CCTV:</p>
                <div class="space-y-3">
                    <input id="input-source" type="text" placeholder="https://xxxx.ngrok-free.app/video" 
                           class="w-full px-3 py-2 text-xs bg-slate-950 border border-slate-700 rounded-lg text-slate-100 focus:outline-none focus:border-indigo-500" />
                    <div class="flex gap-2">
                        <button onclick="setSource()" class="flex-1 bg-indigo-600 hover:bg-indigo-500 text-white font-medium py-2 px-3 rounded-lg text-xs transition">
                            Terapkan Sumber
                        </button>
                        <button onclick="setSimulate()" class="bg-slate-800 hover:bg-slate-700 text-slate-300 py-2 px-3 rounded-lg text-xs transition">
                            Simulasi
                        </button>
                    </div>
                </div>
            </div>

            <!-- Telemetry Metrics Card -->
            <div class="bg-slate-900 border border-slate-800 rounded-xl p-5">
                <h3 class="text-sm font-semibold text-slate-200 mb-4">📊 Telemetri Real-Time</h3>
                <div class="grid grid-cols-2 gap-3 text-center">
                    <div class="p-3 bg-slate-950 rounded-lg border border-slate-800/80">
                        <div class="text-xs text-slate-400">Fokus Menyimak</div>
                        <div id="m-attentive" class="text-xl font-bold text-emerald-400 mt-1">0</div>
                    </div>
                    <div class="p-3 bg-slate-950 rounded-lg border border-slate-800/80">
                        <div class="text-xs text-slate-400">Angkat Tangan</div>
                        <div id="m-hand" class="text-xl font-bold text-amber-400 mt-1">0</div>
                    </div>
                    <div class="p-3 bg-slate-950 rounded-lg border border-slate-800/80">
                        <div class="text-xs text-slate-400">Tertidur / Fatigue</div>
                        <div id="m-sleep" class="text-xl font-bold text-red-400 mt-1">0</div>
                    </div>
                    <div class="p-3 bg-slate-950 rounded-lg border border-slate-800/80">
                        <div class="text-xs text-slate-400">Skor Fokus Kelas</div>
                        <div id="m-focus" class="text-xl font-bold text-indigo-400 mt-1">0%</div>
                    </div>
                </div>
            </div>
        </div>
    </div>

    <script>
        async function fetchTelemetry() {
            try {
                const res = await fetch('/api/telemetry');
                const data = await res.json();
                document.getElementById('m-attentive').innerText = data.attentive_count || 0;
                document.getElementById('m-hand').innerText = data.hand_raised_count || 0;
                document.getElementById('m-sleep').innerText = data.sleeping_count || 0;
                document.getElementById('m-focus').innerText = (data.average_focus_score || 0) + '%';
            } catch(e) {}
        }
        setInterval(fetchTelemetry, 1500);

        async function setSource() {
            const src = document.getElementById('input-source').value.trim();
            if(!src) return alert('Masukkan URL stream!');
            await fetch('/api/set_source', {
                method: 'POST',
                headers: {'Content-Type': 'application/json'},
                body: JSON.stringify({source: src})
            });
            alert('Sumber video berhasil diubah!');
        }

        async function setSimulate() {
            await fetch('/api/set_source', {
                method: 'POST',
                headers: {'Content-Type': 'application/json'},
                body: JSON.stringify({source: 'simulate'})
            });
            alert('Mode simulasi aktif.');
        }
    </script>
</body>
</html>
    """
'''
with open('app.py', 'w') as f:
    f.write(code)
print('✅ File app.py berhasil ditulis!')

In [ ]:
# 4. Jalankan FastAPI & Cloudflare Tunnel (Auto-Reconnect)
!pkill -9 -f cloudflared 2>/dev/null || true
!pkill -9 -f uvicorn 2>/dev/null || true
!nohup uvicorn app:app --host 127.0.0.1 --port 8000 > uvicorn.log 2>&1 &
!nohup cloudflared tunnel --url http://127.0.0.1:8000 --logfile tunnel.log > /dev/null 2>&1 &

import time
import re

print("⏳ Menghubungkan ke Cloudflare Tunnel (tunggu 5-10 detik)...")
tunnel_url = None
for attempt in range(15):
    time.sleep(1.5)
    try:
        with open('tunnel.log', 'r') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                tunnel_url = match.group(0)
                break
    except Exception:
        pass

if tunnel_url:
    print('\n' + '='*65)
    print(f'🚀 LIVE DASHBOARD VISION ENGINE : {tunnel_url}')
    print(f'📹 MJPEG VIDEO STREAM           : {tunnel_url}/video_feed')
    print(f'📊 TELEMETRY API                : {tunnel_url}/api/telemetry')
    print('='*65 + '\n')
else:
    print('⚠️ Isi log tunnel:')
    get_ipython().system('cat tunnel.log | tail -n 15')
